# 🧠 Aryan's Preprocessing Pipeline
## Exact Replication of the RSNA 2025 Brain Aneurysm Detection — 1st Place Solution

This notebook faithfully replicates the **exact preprocessing pipeline** used by the 1st place winner of the RSNA 2025 Intracranial Aneurysm Detection competition.

### ✅ Works on both **Kaggle** and **Local (Windows/Linux/Mac)**
The notebook auto-detects the environment and adjusts paths, tool installation, and temp directories accordingly.

### Pipeline Steps (mirroring the original codebase):
1. **Load all DICOM slices** from a patient folder
2. **Majority-based slice filtering** — retain only slices matching the majority `(Rows, Cols, PixelSpacing)` configuration, and filter outliers in interslice spacing (exactly as in `rsna_dcm2niix.py`)
3. **DICOM → NIfTI conversion** — using `dcm2niix` with `gdcmconv --raw` fallback (exactly as in `convert_dicom_to_nifti()`)
4. **Orientation standardization** — using nnU-Net's `SimpleITKIOWithReorient` with RAS orientation (exactly as in `rsna_submission_roi.py`)
5. **Per-volume Z-score normalization** — using nnU-Net's `ZScoreNormalization` (exactly as in `default_normalization_schemes.py`)

### Key principle:
**Every function and logic path is reused directly from the winner's repository — no redesign or approximation.**

---

## 0. Setup & Imports

We auto-detect the environment (Kaggle vs. local) and configure the repository root accordingly.
All winner imports are then loaded from the repository's `src/` and `nnUNet/` directories.

In [2]:
import sys
import os
import shutil
import tempfile
import json
import subprocess
from pathlib import Path
from collections import Counter

import numpy as np
import pydicom
import nibabel as nib
import SimpleITK as sitk

# ── Detect environment ──
IS_KAGGLE = os.path.exists('/kaggle')

# ── Point to the repository root so all winner imports work ──
if IS_KAGGLE:
    # On Kaggle: the winner's repo must be uploaded as a Kaggle dataset.
    # ╔══════════════════════════════════════════════════════════════╗
    # ║  UPDATE this to match your Kaggle dataset slug             ║
    # ╚══════════════════════════════════════════════════════════════╝
    KAGGLE_DATASET_NAME = "rsna2025-1st-place"  # ← CHANGE if your dataset name differs
    REPO_ROOT = Path(f"/kaggle/input/{KAGGLE_DATASET_NAME}")
    if not REPO_ROOT.exists():
        # Fallback: auto-detect the first dataset in /kaggle/input
        input_dirs = [d for d in Path('/kaggle/input').iterdir() if d.is_dir()]
        if input_dirs:
            REPO_ROOT = input_dirs[0]
            print(f"⚠ Dataset '{KAGGLE_DATASET_NAME}' not found. Auto-detected: {REPO_ROOT.name}")
        else:
            raise FileNotFoundError(
                "No datasets found in /kaggle/input. "
                "Upload the winner's repo as a Kaggle dataset first."
            )
else:
    # Local environment (Windows/Linux/Mac)
    REPO_ROOT = Path(r"c:\Users\maila\Desktop\Major_1st Place solution\rsna2025_1st_place-main")

# ── Add repo root and nnUNet to sys.path for all winner imports ──
for p in [str(REPO_ROOT), str(REPO_ROOT / "nnUNet")]:
    if p not in sys.path:
        sys.path.insert(0, p)

# ── Set working directory to a writable location ──
if IS_KAGGLE:
    os.chdir('/kaggle/working')
else:
    os.chdir(str(REPO_ROOT))

# ── Handle .project-root marker ──
# On Kaggle, the input directory is read-only, so we skip the assertion.
# Instead we just verify that REPO_ROOT has the expected structure.
project_root_marker = REPO_ROOT / ".project-root"
if project_root_marker.exists():
    print("✓ .project-root marker found")
elif (REPO_ROOT / "src" / "my_utils").exists():
    print("Note: .project-root marker not found (expected on Kaggle read-only input).")
    print("  Repo structure verified via src/my_utils/ — continuing.")
else:
    raise FileNotFoundError(
        f"Cannot verify repository structure at {REPO_ROOT}. "
        f"Expected either .project-root or src/my_utils/ to exist."
    )

print(f"Environment          : {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"Repository root      : {REPO_ROOT}")
print(f"Python version       : {sys.version}")
print("Setup complete ✓")

FileNotFoundError: [WinError 3] The system cannot find the path specified: '\\kaggle\\input'

## 0.1 Import Winner's Functions Directly

We import the **exact** functions used by the winner for each preprocessing stage:
- `audit_folder` — determines the majority `(Rows, Cols, PixelSpacing)` configuration
- `prepare_majority_subset` — copies only matching DICOM slices
- `filter_by_slice_spacing` — removes slices with outlier interslice spacing
- `convert_dicom_to_nifti` — full DICOM→NIfTI conversion with dcm2niix + gdcmconv fallback
- `SimpleITKIOWithReorient` — nnU-Net's orientation standardiser (RAS reorientation)
- `ZScoreNormalization` — nnU-Net's per-volume z-score normaliser

In [ ]:
# ── Winner's DICOM audit & filtering functions ──
from src.my_utils.rsna_dcm2niix import (
    audit_folder,
    prepare_majority_subset,
    filter_by_slice_spacing,
    convert_dicom_to_nifti,
    is_dicom_file,
    scan_dicom_files,
    which_or_die,
    convert_to_ras_orientation,
)

# ── Winner's NIfTI loading & RAS conversion ──
from src.my_utils.rsna_utils import (
    load_nifti,
    load_nifti_and_convert_to_ras,
)

# ── nnU-Net's orientation-aware IO (used in the submission pipeline) ──
from nnunetv2.imageio.simpleitk_reader_writer import SimpleITKIOWithReorient

# ── nnU-Net's normalization (ZScoreNormalization is the default for non-CT) ──
from nnunetv2.preprocessing.normalization.default_normalization_schemes import ZScoreNormalization

print("All winner functions imported successfully ✓")

## 0.2 Install & Verify External Tools

The winner's pipeline requires **dcm2niix** and **gdcmconv** on PATH.  
On **Kaggle**, these are installed automatically via `apt-get`.  
On **local machines**, they should already be installed and on PATH.

In [ ]:
# ── Install external tools if on Kaggle ──
if IS_KAGGLE:
    print("Installing dcm2niix and gdcmconv via apt-get...")
    subprocess.run(
        ["apt-get", "update", "-qq"],
        check=True, capture_output=True
    )
    subprocess.run(
        ["apt-get", "install", "-y", "-qq", "dcm2niix", "libgdcm-tools"],
        check=True, capture_output=True
    )
    print("✓ Installation complete")
else:
    print("Local environment: expecting dcm2niix and gdcmconv on PATH")

# ── Check that dcm2niix and gdcmconv are available (exactly as the winner does) ──
try:
    dcm2niix_path = which_or_die("dcm2niix")
    print(f"✓ dcm2niix found: {dcm2niix_path}")
except FileNotFoundError:
    print("✗ dcm2niix NOT found on PATH!")
    print("  Install via: apt-get install dcm2niix (Linux) or conda install -c conda-forge dcm2niix")

try:
    gdcmconv_path = which_or_die("gdcmconv")
    print(f"✓ gdcmconv found: {gdcmconv_path}")
except FileNotFoundError:
    print("✗ gdcmconv NOT found on PATH!")
    print("  Install via: apt-get install libgdcm-tools (Linux) or conda install -c conda-forge gdcm")

## 1. Configure Input Path

Specify the path to a patient folder containing DICOM slices (`.dcm` files).  
The folder should contain all slices for a single series.

On **Kaggle**, paths are relative to the dataset input directory.  
On **local**, paths can be absolute or relative to `REPO_ROOT`.

In [ ]:
# ╔════════════════════════════════════════════════════════════════╗
# ║  SET THIS TO YOUR PATIENT DICOM FOLDER                      ║
# ╚════════════════════════════════════════════════════════════════╝

# The default uses the inputs/ folder inside the repository.
# Both Kaggle and local environments use the same relative structure.
SERIES_UID = "1.2.826.0.1.3680043.8.498.10005158603912009425635473100344077317"
PATIENT_DICOM_DIR = REPO_ROOT / "inputs" / SERIES_UID

# ── If your DICOM data is in a SEPARATE Kaggle dataset, uncomment below ──
# if IS_KAGGLE:
#     PATIENT_DICOM_DIR = Path("/kaggle/input/rsna-dicom-data") / SERIES_UID

assert PATIENT_DICOM_DIR.exists(), f"Patient DICOM folder does not exist: {PATIENT_DICOM_DIR}"
assert PATIENT_DICOM_DIR.is_dir(), f"Path is not a directory: {PATIENT_DICOM_DIR}"

print(f"Patient DICOM folder: {PATIENT_DICOM_DIR}")
print(f"Folder name (SeriesInstanceUID): {PATIENT_DICOM_DIR.name}")

---
## 2. Stage 1 — Load & Audit DICOM Slices

We use the winner's `audit_folder()` to:
- Scan all DICOM files
- Determine the **majority** `(Rows, Cols, PixelSpacing_row, PixelSpacing_col)` across all slices
- Count file-level series composition

This is the first step of the winner's `convert_dicom_to_nifti()` function  
(see `rsna_dcm2niix.py` lines 1390‒1395).

In [ ]:
# ── Exactly as the winner does: audit the folder ──
FILE_EXTS = (".dcm", "")  # Same as RunConfig default

audit = audit_folder(PATIENT_DICOM_DIR, FILE_EXTS)

print("=" * 60)
print("STAGE 1: DICOM Folder Audit")
print("=" * 60)
print(f"Total files in folder  : {audit.total_files}")
print(f"DICOM files detected   : {audit.dicom_files}")
print(f"Majority configuration : {audit.majority_key}")
if audit.majority_key:
    rows, cols, ps_row, ps_col = audit.majority_key
    print(f"  → Rows × Cols        : {rows} × {cols}")
    print(f"  → PixelSpacing       : ({ps_row}, {ps_col}) mm")
print(f"Series composition     : {len(audit.series_map)} series")
for (pid, suid), count in sorted(audit.series_map.items(), key=lambda x: -x[1]):
    print(f"  PatientID={pid}, SeriesUID=...{suid[-20:]}: {count} files")

# ── Validation checks ──
assert audit.dicom_files > 0, "No DICOM files found in the folder!"
assert audit.majority_key is not None, "Could not determine majority configuration!"
print("\n✓ Audit passed: DICOM files found with valid majority configuration")

## 3. Stage 2 — Majority-Based Slice Filtering

The winner filters out scout/localizer images and slices with different acquisition parameters by:
1. Keeping only DICOMs matching the majority `(Rows, Cols, PixelSpacing)` — `prepare_majority_subset()`
2. Removing slices with **outlier interslice spacing** (>2× or <0.5× the median) — `filter_by_slice_spacing()`

This is done with `use_majority_size=True` and `use_slice_spacing_filter=True`  
(see `rsna_dcm2niix.py` lines 1405‒1422).

In [ ]:
# ── Create temp directory for filtered DICOM subset ──
# The winner creates a temp dir and copies only majority-matching slices
# On Kaggle, /tmp is writable but /kaggle/working is preferred for visibility
if IS_KAGGLE:
    _tmp_base = Path("/kaggle/working/tmp")
    _tmp_base.mkdir(parents=True, exist_ok=True)
    tmp_root = Path(tempfile.mkdtemp(prefix="aryan_preproc_", dir=str(_tmp_base)))
else:
    tmp_root = Path(tempfile.mkdtemp(prefix="aryan_preproc_"))

majority_subset_dir = Path(tempfile.mkdtemp(dir=str(tmp_root)))

print("=" * 60)
print("STAGE 2: Majority-Based Slice Filtering")
print("=" * 60)

# ── Use the winner's exact function with the same default parameters ──
n_copied, slice_filter_log = prepare_majority_subset(
    src_dir=PATIENT_DICOM_DIR,
    dst_dir=majority_subset_dir,
    majority_key=audit.majority_key,
    copy_mode="auto",                    # Same as RunConfig.copy_mode
    pixel_spacing_precision=2,           # Same as audit default
    slice_spacing_filter=True,           # Same as RunConfig.use_slice_spacing_filter
    slice_spacing_tolerance=2.0,         # Same as RunConfig.slice_spacing_tolerance
)

print(f"Original DICOM files   : {audit.dicom_files}")
print(f"After majority filter  : {n_copied} files")
print(f"Slices removed         : {audit.dicom_files - n_copied}")
if slice_filter_log:
    print(f"Spacing filter log     : {slice_filter_log}")
else:
    print(f"Spacing filter log     : No outliers detected")

# ── Validation checks ──
assert n_copied > 0, "No slices remained after filtering!"
filtered_files = list(majority_subset_dir.iterdir())
assert len(filtered_files) == n_copied, f"File count mismatch: {len(filtered_files)} vs {n_copied}"
print(f"\n✓ Filtering passed: {n_copied} slices retained in temp directory")

## 4. Stage 3 — DICOM → NIfTI Conversion

We use the winner's `convert_dicom_to_nifti()` which:
1. Runs `dcm2niix` with flags `("-z", "y", "-b", "y", "-i", "n", "-f", "%s")`
2. On failure, falls back to `gdcmconv --raw` → re-run `dcm2niix`
3. Handles multi-file output (selects primary NIfTI)
4. Normalizes 4D → 3D if needed
5. Converts to RAS orientation via `nib.as_closest_canonical()`

Note: In the **submission pipeline** (`rsna_submission_roi.py` line 806), `convert_to_ras=False` is used  
because orientation is handled later by `SimpleITKIOWithReorient`. For the standalone preprocessing  
(batch conversion `rsna_dcm2niix.py`), `convert_to_ras=True` is the default.

We replicate the **submission pipeline** path: `convert_to_ras=False`, then apply `SimpleITKIOWithReorient`  
separately—exactly as `_predict_probs_with_pipeline()` does.

In [ ]:
# ── Output directory for NIfTI ──
# On Kaggle, output must go to /kaggle/working (writable)
# On local, output goes to nifiti_outputs/ inside the repo
if IS_KAGGLE:
    nifti_out_dir = Path("/kaggle/working/nifti_outputs")
else:
    nifti_out_dir = REPO_ROOT / "nifiti_outputs"

nifti_out_dir.mkdir(parents=True, exist_ok=True)


print("=" * 60)
print("STAGE 3: DICOM → NIfTI Conversion (dcm2niix + gdcmconv fallback)")
print("=" * 60)

# ── Use the winner's exact conversion function ──
# Mirroring the submission pipeline: convert_to_ras=False
# (RAS is applied later via SimpleITKIOWithReorient)
nifti_path, used_dir, conv_logs, conv_errors = convert_dicom_to_nifti(
    dir_path=PATIENT_DICOM_DIR,
    out_dir=nifti_out_dir,
    tmp_root=tmp_root,
    use_majority_size=True,              # Same as submission
    copy_mode="auto",                    # Same as submission
    file_exts=(".dcm", ""),              # Same as submission
    dcm2niix_flags=("-z", "y", "-b", "y", "-i", "n", "-f", "%s"),  # Winner's default flags
    gdcm_first=False,                    # Same as submission
    use_slice_spacing_filter=True,        # Same as submission
    slice_spacing_tolerance=2.0,          # Same as submission
    convert_to_ras=False,                 # Submission pipeline uses False; RAS via SimpleITKIOWithReorient
)

# Print conversion logs
print("\nConversion logs:")
for log in conv_logs:
    print(f"  {log}")

if conv_errors:
    print("\n⚠ Conversion errors:")
    for err in conv_errors:
        print(f"  {err}")

# ── Validation checks ──
assert nifti_path is not None, f"NIfTI conversion failed! Errors: {conv_errors}"
assert Path(nifti_path).exists(), f"Generated NIfTI does not exist: {nifti_path}"

print(f"\n✓ NIfTI generated: {nifti_path}")
print(f"  File size: {Path(nifti_path).stat().st_size / 1024:.1f} KB")

### 4.1 Validate NIfTI Intermediate Output

Before proceeding, verify the generated NIfTI has valid properties.

In [ ]:
# ── Quick inspection of the generated NIfTI ──
nii_check = nib.load(str(nifti_path))
nii_shape = nii_check.shape
nii_zooms = nii_check.header.get_zooms()
nii_dtype = nii_check.header.get_data_dtype()

print("NIfTI intermediate validation:")
print(f"  Shape      : {nii_shape}")
print(f"  Spacing    : {nii_zooms}")
print(f"  Data type  : {nii_dtype}")
print(f"  Dimensions : {len(nii_shape)}D")

# ── Validation ──
assert len(nii_shape) == 3, f"Expected 3D volume, got {len(nii_shape)}D with shape {nii_shape}"
assert all(s > 0 for s in nii_shape), f"Invalid shape: {nii_shape}"
assert all(z > 0 for z in nii_zooms[:3]), f"Invalid spacing: {nii_zooms}"
print("\n✓ NIfTI intermediate output is valid")

## 5. Stage 4 — Orientation Standardization (nnU-Net SimpleITKIOWithReorient)

The winner uses nnU-Net's `SimpleITKIOWithReorient` to load and reorient volumes to **RAS** orientation.  
This is the exact path used in `rsna_submission_roi.py` → `RsnaRoiPipeline.load_nifti_volume()`  
(line 877: `io_handler.read_images([nifti_path], orientation="RAS")`).

The `SimpleITKIOWithReorient.read_images()` method:
1. Reads the NIfTI via SimpleITK
2. Applies `sitk.DICOMOrient(image, "RAS")` to reorient
3. Returns the volume as `(C, Z, Y, X)` with spacing in `(Z, Y, X)` order

In [ ]:
print("=" * 60)
print("STAGE 4: Orientation Standardization (SimpleITKIOWithReorient → RAS)")
print("=" * 60)

# ── Use the exact nnU-Net IO handler as the winner ──
io_handler = SimpleITKIOWithReorient()
image_nnunet, properties = io_handler.read_images([str(nifti_path)], orientation="RAS")

# image_nnunet shape: (C, Z, Y, X)  — C=1 for single-channel
# properties['spacing'] is in (Z, Y, X) order for nnU-Net

print(f"Volume shape (C,Z,Y,X) : {image_nnunet.shape}")
print(f"Volume dtype           : {image_nnunet.dtype}")
print(f"Spacing (Z,Y,X) mm     : {properties['spacing']}")

sitk_stuff = properties.get('sitk_stuff', {})
if 'original_orientation' in sitk_stuff:
    print(f"Original orientation   : {sitk_stuff['original_orientation']}")
print(f"SITK spacing (X,Y,Z)   : {sitk_stuff.get('spacing', 'N/A')}")
print(f"SITK origin            : {sitk_stuff.get('origin', 'N/A')}")

# ── Validation checks ──
assert image_nnunet.ndim == 4, f"Expected 4D (C,Z,Y,X), got {image_nnunet.ndim}D"
assert image_nnunet.shape[0] == 1, f"Expected 1 channel, got {image_nnunet.shape[0]}"
assert not np.isnan(image_nnunet).any(), "NaN values detected after orientation standardization!"
assert not np.isinf(image_nnunet).any(), "Inf values detected after orientation standardization!"

spacing_zyx = properties['spacing']
assert len(spacing_zyx) == 3, f"Expected 3 spacing values, got {len(spacing_zyx)}"
assert all(s > 0 for s in spacing_zyx), f"Invalid spacing: {spacing_zyx}"

print("\n✓ Orientation standardization passed: volume is in RAS orientation")

### 5.1 Validate Orientation Consistency

Verify the orientation is consistent with the original pipeline by checking the transformed spacing  
matches expected anatomical conventions.

In [ ]:
# ── Cross-check: read the same file with raw SimpleITK and verify RAS ──
itk_raw = sitk.ReadImage(str(nifti_path))
original_orient = sitk.DICOMOrientImageFilter_GetOrientationFromDirectionCosines(itk_raw.GetDirection())
itk_ras = sitk.DICOMOrient(itk_raw, "RAS")
ras_orient = sitk.DICOMOrientImageFilter_GetOrientationFromDirectionCosines(itk_ras.GetDirection())

print("Orientation consistency check:")
print(f"  Original orientation  : {original_orient}")
print(f"  After DICOMOrient RAS : {ras_orient}")
print(f"  Was reorientation needed: {original_orient != 'RAS'}")

# Verify shape matches what SimpleITKIOWithReorient returned
ras_array = sitk.GetArrayFromImage(itk_ras)  # Returns (Z, Y, X)
print(f"  SimpleITK RAS shape   : {ras_array.shape} (Z,Y,X)")
print(f"  nnU-Net output shape  : {image_nnunet.shape} (C,Z,Y,X)")

assert ras_array.shape == image_nnunet.shape[1:], (
    f"Shape mismatch: SimpleITK {ras_array.shape} vs nnU-Net {image_nnunet.shape[1:]}"
)
print("\n✓ Orientation consistency verified")

## 6. Stage 5 — Per-Volume Z-Score Normalization

The winner applies nnU-Net's standard **per-volume z-score normalization**:

```
normalized = (image - mean) / max(std, 1e-8)
```

This is implemented in `ZScoreNormalization.run()` from  
`nnunetv2/preprocessing/normalization/default_normalization_schemes.py` (lines 27‒50).

In the winner's context:
- The normalization is embedded in nnU-Net's preprocessing pipeline  
  (called via `AdaptiveSparsePredictor` during vessel segmentation inference)
- For brain angiography data, `use_mask_for_norm=False` is typically used  
  (i.e., compute mean/std over the **entire** volume, not just the brain mask)
- The `intensityproperties` dict is required by the ABC but not used by ZScore  
  (it only needs mean/std computed on-the-fly)

In [ ]:
print("=" * 60)
print("STAGE 5: Per-Volume Z-Score Normalization (nnU-Net's ZScoreNormalization)")
print("=" * 60)

# ── Pre-normalization statistics ──
volume_pre = image_nnunet[0].copy()  # Extract single channel (Z, Y, X)
print("\nPre-normalization statistics:")
print(f"  Shape        : {volume_pre.shape}")
print(f"  Dtype        : {volume_pre.dtype}")
print(f"  Min          : {volume_pre.min():.4f}")
print(f"  Max          : {volume_pre.max():.4f}")
print(f"  Mean         : {volume_pre.mean():.4f}")
print(f"  Std          : {volume_pre.std():.4f}")
print(f"  Median       : {np.median(volume_pre):.4f}")

# ── Apply the winner's exact normalization ──
# ZScoreNormalization requires intensityproperties (can be empty dict for z-score)
# use_mask_for_norm=False → compute mean/std over entire volume
normalizer = ZScoreNormalization(
    use_mask_for_norm=False,
    intensityproperties={},  # Not used by ZScore but required by ABC
    target_dtype=np.float32,
)

# The .run() method normalizes the volume in-place and returns it
volume_normalized = normalizer.run(volume_pre.copy(), seg=None)

# ── Post-normalization statistics ──
print("\nPost-normalization statistics:")
print(f"  Shape        : {volume_normalized.shape}")
print(f"  Dtype        : {volume_normalized.dtype}")
print(f"  Min          : {volume_normalized.min():.4f}")
print(f"  Max          : {volume_normalized.max():.4f}")
print(f"  Mean         : {volume_normalized.mean():.6f}   (should be ≈ 0)")
print(f"  Std          : {volume_normalized.std():.6f}    (should be ≈ 1)")
print(f"  Median       : {np.median(volume_normalized):.4f}")

# ── Validation checks ──
assert volume_normalized.dtype == np.float32, f"Expected float32, got {volume_normalized.dtype}"
assert not np.isnan(volume_normalized).any(), "NaN values after normalization!"
assert not np.isinf(volume_normalized).any(), "Inf values after normalization!"
assert abs(volume_normalized.mean()) < 1e-5, (
    f"Mean should be ~0 after z-score, got {volume_normalized.mean():.6f}"
)
assert abs(volume_normalized.std() - 1.0) < 1e-5, (
    f"Std should be ~1 after z-score, got {volume_normalized.std():.6f}"
)
print("\n✓ Z-score normalization passed: mean≈0, std≈1")

### 6.1 Verify Normalization Matches Manual Z-Score

Cross-check that the nnU-Net normalizer produces the same result as manual z-score computation.

In [ ]:
# ── Manual z-score as cross-check ──
volume_manual = image_nnunet[0].copy().astype(np.float32)
manual_mean = volume_manual.mean()
manual_std = volume_manual.std()
volume_manual -= manual_mean
volume_manual /= max(manual_std, 1e-8)

# ── Compare ──
max_diff = np.abs(volume_normalized - volume_manual).max()
print(f"Manual z-score cross-check:")
print(f"  Max absolute difference : {max_diff:.2e}")
print(f"  Mean absolute difference: {np.abs(volume_normalized - volume_manual).mean():.2e}")

assert max_diff < 1e-5, f"Normalization mismatch! Max diff = {max_diff}"
print("\n✓ Normalization cross-check passed: nnU-Net ZScore matches manual computation")

---
## 7. Final Output Summary

Print the fully processed volume's shape, dtype, and statistics.  
The output is compatible with the winner's downstream pipeline:
- **Vessel segmentation** (nnU-Net adaptive sparse search predictor)
- **ROI extraction** and **ROI classification**

In [ ]:
D, H, W = volume_normalized.shape

print("\n" + "═" * 70)
print("  FINAL PREPROCESSED VOLUME SUMMARY")
print("═" * 70)
print(f"")
print(f"  Shape (D, H, W)       : ({D}, {H}, {W})")
print(f"  Total voxels          : {D * H * W:,}")
print(f"  Data type             : {volume_normalized.dtype}")
print(f"  Spacing (Z, Y, X) mm  : ({spacing_zyx[0]:.4f}, {spacing_zyx[1]:.4f}, {spacing_zyx[2]:.4f})")
print(f"  Physical size (mm)    : ({D * spacing_zyx[0]:.1f}, {H * spacing_zyx[1]:.1f}, {W * spacing_zyx[2]:.1f})")
print(f"")
print(f"  ── Intensity Statistics ──")
print(f"  Min                   : {volume_normalized.min():.4f}")
print(f"  Max                   : {volume_normalized.max():.4f}")
print(f"  Mean                  : {volume_normalized.mean():.6f}")
print(f"  Std                   : {volume_normalized.std():.6f}")
print(f"  Percentile  1%        : {np.percentile(volume_normalized, 1):.4f}")
print(f"  Percentile 99%        : {np.percentile(volume_normalized, 99):.4f}")
print(f"")
print(f"  ── Pipeline Properties ──")
print(f"  Original DICOM files  : {audit.dicom_files}")
print(f"  After filtering       : {n_copied}")
print(f"  Orientation           : RAS (standardized via SimpleITKIOWithReorient)")
print(f"  Normalization         : Per-volume Z-score (mean=0, std=1)")
print(f"  Contains NaN          : {np.isnan(volume_normalized).any()}")
print(f"  Contains Inf          : {np.isinf(volume_normalized).any()}")
print(f"")
print("═" * 70)
print(f"  ✓ Output is fully compatible with downstream vessel segmentation")
print(f"    and ROI extraction models from the 1st place solution.")
print("═" * 70)

## 8. Final Validation Summary

Run all pipeline consistency checks in one cell.

In [ ]:
print("=" * 60)
print("FINAL PIPELINE VALIDATION")
print("=" * 60)

checks = []

# 1. Dimensionality
dim_ok = volume_normalized.ndim == 3
checks.append(("3D volume", dim_ok, f"ndim={volume_normalized.ndim}"))

# 2. Valid spacing
sp_ok = all(s > 0 for s in spacing_zyx)
checks.append(("Valid spacing", sp_ok, f"spacing={spacing_zyx}"))

# 3. No NaN
nan_ok = not np.isnan(volume_normalized).any()
checks.append(("No NaN values", nan_ok, ""))

# 4. No Inf
inf_ok = not np.isinf(volume_normalized).any()
checks.append(("No Inf values", inf_ok, ""))

# 5. Float32 dtype
dtype_ok = volume_normalized.dtype == np.float32
checks.append(("float32 dtype", dtype_ok, f"dtype={volume_normalized.dtype}"))

# 6. Z-score mean ≈ 0
mean_ok = abs(volume_normalized.mean()) < 1e-4
checks.append(("Z-score mean ≈ 0", mean_ok, f"mean={volume_normalized.mean():.6f}"))

# 7. Z-score std ≈ 1
std_ok = abs(volume_normalized.std() - 1.0) < 1e-4
checks.append(("Z-score std ≈ 1", std_ok, f"std={volume_normalized.std():.6f}"))

# 8. Reasonable volume size
size_ok = all(s >= 10 for s in volume_normalized.shape)
checks.append(("Reasonable volume size", size_ok, f"shape={volume_normalized.shape}"))

# 9. Uses nnU-Net conventions (spacing order)
conv_ok = len(spacing_zyx) == 3
checks.append(("nnU-Net spacing convention", conv_ok, f"len={len(spacing_zyx)}"))

# 10. Consistent with submission pipeline
pipeline_ok = image_nnunet.shape[0] == 1  # Single channel
checks.append(("Single channel (C=1)", pipeline_ok, f"C={image_nnunet.shape[0]}"))

all_passed = True
for name, passed, detail in checks:
    status = "✓" if passed else "✗"
    detail_str = f" ({detail})" if detail else ""
    print(f"  {status}  {name}{detail_str}")
    if not passed:
        all_passed = False

print()
if all_passed:
    print("══════════════════════════════════════════════════════════")
    print("  ✓ ALL CHECKS PASSED — Pipeline output is valid!")
    print("══════════════════════════════════════════════════════════")
else:
    print("══════════════════════════════════════════════════════════")
    print("  ✗ SOME CHECKS FAILED — Please review above")
    print("══════════════════════════════════════════════════════════")

## Visulalization

In [ ]:
import nibabel as nib
import matplotlib.pyplot as plt
from ipywidgets import interact

# Load
img = nib.load(nifti_path)
data = img.get_fdata()

def show_slice(z):
    plt.figure(figsize=(5,5))
    plt.imshow(data[z, :, :], cmap='gray')
    plt.title(f"Slice {z}")
    plt.axis('off')
    plt.show()

interact(show_slice, z=(0, data.shape[0]-1))

## 9. Cleanup

Remove temporary files created during preprocessing.

In [ ]:
# ── Clean up temporary directory ──
try:
    shutil.rmtree(tmp_root, ignore_errors=True)
    print(f"✓ Cleaned up temp directory: {tmp_root}")
except Exception as e:
    print(f"⚠ Cleanup failed: {e}")

---
## 10. Appendix — Pipeline Architecture Reference

### Functions Used (from the winner's codebase)

| Stage | Function | Source File |
|-------|----------|-------------|
| Audit | `audit_folder()` | `src/my_utils/rsna_dcm2niix.py` |
| Filter | `prepare_majority_subset()` | `src/my_utils/rsna_dcm2niix.py` |
| Spacing | `filter_by_slice_spacing()` | `src/my_utils/rsna_dcm2niix.py` |
| Convert | `convert_dicom_to_nifti()` | `src/my_utils/rsna_dcm2niix.py` |
| Orient | `SimpleITKIOWithReorient.read_images()` | `nnUNet/nnunetv2/imageio/simpleitk_reader_writer.py` |
| Normalize | `ZScoreNormalization.run()` | `nnUNet/nnunetv2/preprocessing/normalization/default_normalization_schemes.py` |

### Winner's Inference Pipeline Flow

```
DICOM folder
    │
    ├── [1] audit_folder() ──────────── Majority (Rows, Cols, PixelSpacing)
    ├── [2] prepare_majority_subset() ─ Filter outlier slices
    ├── [3] convert_dicom_to_nifti() ── dcm2niix (+gdcmconv fallback)
    ├── [4] SimpleITKIOWithReorient ──── RAS orientation standardization
    ├── [5] ZScoreNormalization ──────── Per-volume z-score
    │
    └── Preprocessed volume (D, H, W) float32, mean≈0, std≈1
        │
        ├── → Vessel Segmentation (nnU-Net sparse search)
        ├── → ROI Extraction
        └── → ROI Classification (13 locations + Aneurysm Present)
```